In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
import torch
from transformers import pipeline

C:\Users\micha\anaconda3\envs\transformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pipe_base = pipeline(
    "text-generation",
    model="openai-community/gpt2",
    device="cuda",
)

prompt = "Once upon a time there was a fairy"

baseline = pipe_base(
    prompt,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.9,
    top_p=0.95
)

baseline = baseline[0]["generated_text"]
print(baseline)

Device set to use cuda
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time there was a fairy-tale on the wall saying, "One day I will be a princess, and another tomorrow I will be a boy." My father and I had been discussing the topic. The girl said that she had been raised by a fairy-tale queen, but only two days before that day she had left a party to visit her sister, who was also a princess. We heard the news that she had disappeared. When we went to check on her we heard a familiar noise. We were surprised. What had happened? Did she disappear or did she disappear at all? When I saw the young girl, I thought she was still there. It felt very strange. What had happened to her? It was a little scary for me. I knew I had to ask her. It made me wonder what had happened to her. Even after we left, how had she gone about leaving?

We went to the place where she left, where I picked her up and we went back to the house.


In [3]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset('vicclab/fairy_tales')

In [4]:
#small_ds = dataset["train"].train_test_split(
#    test_size=0.9, seed=42)

In [5]:
#train_val = small_ds["train"].train_test_split(
#    test_size=0.2, seed=42)

In [6]:
train_val = dataset["train"].train_test_split(
    test_size=0.2, seed=42)

In [7]:
dataset = DatasetDict({
    "train": train_val["train"],
    "validation": train_val["test"]
})

In [8]:
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Train size: 82878
Validation size: 20720


In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize_function(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256   # ↑ important: longer context
    )
    return enc

In [11]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map: 100%|█████████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 34900.26 examples/s]


In [12]:
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

Filter: 100%|██████████████████████████████████████████████████████████| 20720/20720 [00:00<00:00, 46368.16 examples/s]


In [13]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [14]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2"
).to("cuda")

training_args = TrainingArguments(
    output_dir="models/results",
    eval_strategy="epoch",
    num_train_epochs=5,              # ↑ more signal at least 5
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="models/logs",
    save_strategy="epoch",
    report_to="none"
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.412700,3.552787
2,3.177700,3.476668
3,2.904400,3.489201
4,2.717600,3.526197
5,2.532700,3.583601


TrainOutput(global_step=83275, training_loss=2.977680019079762, metrics={'train_runtime': 10073.2578, 'train_samples_per_second': 33.067, 'train_steps_per_second': 8.267, 'total_flos': 3373994603520000.0, 'train_loss': 2.977680019079762, 'epoch': 5.0})

---

In [18]:
checkpoint_path_16655 = "models/results/checkpoint-16655"

In [19]:
tokenizer_16655 = AutoTokenizer.from_pretrained(checkpoint_path)
model_16655 = AutoModelForCausalLM.from_pretrained(checkpoint_path).to("cuda")

In [20]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_16655,
    tokenizer=tokenizer_16655,
    device="cuda"
)

Device set to use cuda


In [22]:
ft_output_16655 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_16655 = ft_output_16655[0]["generated_text"]
print(finetuned_16655)

Once upon a time there was a fairy king who lived in the forest; and he loved his master very much, but still he always had to keep his word. When day came, he saw a great fire burning near, and he could not get through the wood. But when night came, all went quiet and the light at his mother’s house. So he set out to see her and, with one of his big horns on the fire, got up and took the axe, and the axe out; then he took a little nut that she could but she could only give him a penny and three shillings. When he came back, it was the same story over again, but no one could be sure what was his son’d best doing. But after he had done this, it seemed that a pretty little creature was waiting for him at every house, and so he set off home again, thinking that he would find some work. He had got enough for one head, and they had a little girl


---

In [23]:
checkpoint_path_33310 = "models/results/checkpoint-33310"

In [24]:
tokenizer_33310 = AutoTokenizer.from_pretrained(checkpoint_path)
model_33310 = AutoModelForCausalLM.from_pretrained(checkpoint_path).to("cuda")

In [25]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_33310,
    tokenizer=tokenizer_33310,
    device="cuda"
)

Device set to use cuda


In [27]:
ft_output_33310 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_33310 = ft_output_33310[0]["generated_text"]
print(finetuned_33310)

Once upon a time there was a fairy queen, who had to go on the shore all day long to see her son. She was very happy, and he had a good time, for him, but at last she was gone. The king was not happy as she could be; she was afraid to let him be king of some one, and she thought he was afraid that he must tell her what he liked. So he left her, and went and got a great horse, and got back into a little wood. Then he took out his cock, and cut the head out of it, and so he knew that the head was no better off than it before, and they set out together to a tree to see what was going on. The old woman said, “My name is Gretel! I have only three boys to play here!” They both took out their bibs and rode away to the door, and with the best thing they could get she was a good long day's work. So when


---

In [29]:
checkpoint_path_49965 = "models/results/checkpoint-49965"

In [30]:
tokenizer_49965 = AutoTokenizer.from_pretrained(checkpoint_path)
model_49965 = AutoModelForCausalLM.from_pretrained(checkpoint_path).to("cuda")

In [31]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_49965,
    tokenizer=tokenizer_49965,
    device="cuda"
)

Device set to use cuda


In [32]:
ft_output_49965 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_49965 = ft_output_49965[0]["generated_text"]
print(finetuned_49965)

Once upon a time there was a fairy, a tall woman in the middle of the wood. She had a long white hair, but not much on her head, and her eyes were all green and bright and sharp. Her mother was very kind to her, and loved to talk, and as she spoke she called out: "Are you the son's father?" And when she got home with the child, she was the son. She loved to see the boy, so much, and he was very much loved to her that she sent him away. The boy did so: "I am the son," and then she gave him a box of gold, and a golden apple. Then, as she was sitting, thinking about it, she said to her mother: "Hear a little while I will sit here; and then we will see how it is." So she went out and looked about for a golden penny, and made up the box, and went inside the room, and hid the golden penny in the drawer, and


---

In [33]:
checkpoint_path_66620 = "models/results/checkpoint-66620"

In [34]:
tokenizer_66620 = AutoTokenizer.from_pretrained(checkpoint_path)
model_66620 = AutoModelForCausalLM.from_pretrained(checkpoint_path).to("cuda")

In [35]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_66620,
    tokenizer=tokenizer_66620,
    device="cuda"
)

Device set to use cuda


In [36]:
ft_output_66620 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_66620 = ft_output_66620[0]["generated_text"]
print(finetuned_66620)

Once upon a time there was a fairy named Ilse, who lived in the beautiful castle in the middle of the wood. This very day she lived in a garden and ate flowers, and gave it to the children of her little children as she pleased, and the little ones to play with. When Ilcess wanted, for the old people, she was always nice, and lovely; so when she got so fond they got what she had. The Fairy, for you know, is still the most handsome, and is always willing too, was the most lovely mother. So long ago was the mother of all these pretty babies, that at the end-of-the-day they were called a young girl, and I can't say how she got on. She had her pretty little hands at the bottom of the house, and she was as nice and warm as all the boys, and would sit and play with a little goat, and a great deal of love for the children. Her big feet and long tail seemed to have much


---

In [37]:
checkpoint_path_83275 = "models/results/checkpoint-83275"

In [38]:
tokenizer_83275 = AutoTokenizer.from_pretrained(checkpoint_path)
model_83275 = AutoModelForCausalLM.from_pretrained(checkpoint_path).to("cuda")

In [39]:
pipe_ckpt = pipeline(
    "text-generation",
    model=model_83275,
    tokenizer=tokenizer_83275,
    device="cuda"
)

Device set to use cuda


In [40]:
ft_output_83275 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
)

finetuned_83275 = ft_output_83275[0]["generated_text"]
print(finetuned_83275)

Once upon a time there was a fairy who lived in an inn, where many people gathered to eat and drink. He was very handsome and had many good things; he always put up his eye, but one of these was the water of a bird, which was too much. It had an eye on his head; he was one hour a day, which he ate up at once. The other day, he went out to play with the fish, and had his own head on the pot, and the other the other night to the barn-door, so that all was well. So when the fish came home, he had the fish; but after he had eaten it he was so angry that the cock in the hen's mouth and head began to beat him, and if they hadn't done for him, he would have been hanged. He had to run away before the cock went to the king, and he had two sons, and one was in his own skin, and wanted the pig, who was fat; he had


---

## Lexical Diversity

Higher = more varied phrasing

Distinct-n measures lexical diversity by computing the proportion of unique n-word sequences (n-grams) in a text, where higher values indicate less repetitive phrasing but do not capture semantic quality or true creativity, making it most useful for relative comparison (e.g., before vs. after fine-tuning) on texts of similar length.

In [42]:
def distinct_n(text, n=2):
    tokens = text.lower().split()
    ngrams = list(zip(*[tokens[i:] for i in range(n)]))
    return len(set(ngrams)) / max(1, len(ngrams))

In [43]:
base_text = baseline #baseline[0]["generated_text"]
ft_text_16655 = finetuned_16655 #ft_output[0]["generated_text"]
ft_text_33310 = finetuned_33310 #ft_output[0]["generated_text"]
ft_text_49965 = finetuned_49965 #ft_output[0]["generated_text"]
ft_text_66620 = finetuned_66620 #ft_output[0]["generated_text"]
ft_text_83275 = finetuned_83275 #ft_output[0]["generated_text"]

print("Distinct-2 (base):", distinct_n(base_text, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_16655, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_33310, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_49965, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_66620, 2))
print("Distinct-2 (ft):  ", distinct_n(ft_text_83275, 2))

Distinct-2 (base): 0.8813559322033898
Distinct-2 (ft):   0.9606741573033708
Distinct-2 (ft):   0.9382022471910112
Distinct-2 (ft):   0.9482758620689655
Distinct-2 (ft):   0.9488636363636364
Distinct-2 (ft):   0.9329608938547486


---

## Surprisal (novelty vs generic English)

Slightly higher surprisal after fine-tuning = more novelty  
Too high = incoherent

This code measures surprisal by computing the model’s average negative log-likelihood (cross-entropy loss) over all tokens in the given text, which reflects how unexpected the text is to the model: higher loss means the model assigns lower probability to the observed tokens (higher surprisal), while lower loss means the text is more predictable according to the model.

In [51]:
def surprisal(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return loss.item()

base_surprisal = surprisal(pipe_base.model, pipe_base.tokenizer, base_text)
ft_surprisal_16655   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_16655)
ft_surprisal_33310   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_33310)
ft_surprisal_49965   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_49965)
ft_surprisal_66620   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_66620)
ft_surprisal_83275   = surprisal(pipe_base.model, pipe_base.tokenizer, ft_text_83275)

print("Surprisal (base):", base_surprisal)
print("Surprisal (ft):  ", ft_surprisal_16655)
print("Surprisal (ft):  ", ft_surprisal_33310)
print("Surprisal (ft):  ", ft_surprisal_49965)
print("Surprisal (ft):  ", ft_surprisal_66620)
print("Surprisal (ft):  ", ft_surprisal_83275)

Surprisal (base): 2.225379467010498
Surprisal (ft):   3.1452789306640625
Surprisal (ft):   3.066494941711426
Surprisal (ft):   2.6303486824035645
Surprisal (ft):   3.24499773979187
Surprisal (ft):   2.9723851680755615


---

## Style alignment (fairy-tale vocabulary)

This code measures style alignment by counting how many distinct, predefined fairy-tale–related keywords appear in the text, using their presence as a simple proxy for how closely the text matches a fairy-tale style, without considering context, frequency, or deeper narrative structure.

In [47]:
fairy_words = {"fairy", "king", "queen", "magic", "forest", "spell", "castle"}

def fairy_score(text):
    tokens = set(text.lower().split())
    return len(tokens & fairy_words)

print("Fairy score (base):", fairy_score(base_text))
print("Fairy score (ft):  ", fairy_score(ft_text_16655))
print("Fairy score (ft):  ", fairy_score(ft_text_33310))
print("Fairy score (ft):  ", fairy_score(ft_text_49965))
print("Fairy score (ft):  ", fairy_score(ft_text_66620))
print("Fairy score (ft):  ", fairy_score(ft_text_83275))

Fairy score (base): 0
Fairy score (ft):   2
Fairy score (ft):   2
Fairy score (ft):   0
Fairy score (ft):   2
Fairy score (ft):   1


---

## Self-BLEU

Self-BLEU measures how similar generated samples are to each other. Lower Self-BLEU is better for creativity/diversity 

In [54]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [55]:
def self_bleu(texts, n_gram=2):
    """
    texts: list of generated strings
    n_gram: BLEU-n (2 or 3 recommended)
    """
    smoothie = SmoothingFunction().method1
    scores = []

    weights = {
        2: (0.5, 0.5),
        3: (1/3, 1/3, 1/3),
        4: (0.25, 0.25, 0.25, 0.25)
    }[n_gram]

    tokenized = [t.lower().split() for t in texts]

    for i, hypothesis in enumerate(tokenized):
        references = tokenized[:i] + tokenized[i+1:]
        score = sentence_bleu(
            references,
            hypothesis,
            weights=weights,
            smoothing_function=smoothie
        )
        scores.append(score)

    return sum(scores) / len(scores)

In [56]:
def generate_samples(pipe, prompt, n=30):
    return [
        pipe(
            prompt,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.9,
            top_p=0.95
        )[0]["generated_text"]
        for _ in range(n)
    ]

In [57]:
pipe_16655 = pipeline(
    "text-generation",
    model=model_16655,
    tokenizer=tokenizer_16655,
    device="cuda"
)

Device set to use cuda


In [58]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_16655, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=2))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=2))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `

Baseline Self-BLEU: 0.49504901440416776
Finetuned Self-BLEU: 0.6555706571506238


In [59]:
pipe_33310 = pipeline(
    "text-generation",
    model=model_33310,
    tokenizer=tokenizer_33310,
    device="cuda"
)

Device set to use cuda


In [60]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_33310, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=2))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=2))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.5035257744142648
Finetuned Self-BLEU: 0.6309920474766183


In [61]:
pipe_49965 = pipeline(
    "text-generation",
    model=model_49965,
    tokenizer=tokenizer_49965,
    device="cuda"
)

Device set to use cuda


In [62]:
baseline_texts = generate_samples(pipe_base, prompt, n=30)
finetuned_texts = generate_samples(pipe_49965, prompt, n=30)

print("Baseline Self-BLEU:", self_bleu(baseline_texts, n_gram=2))
print("Finetuned Self-BLEU:", self_bleu(finetuned_texts, n_gram=2))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Baseline Self-BLEU: 0.5080662663264417
Finetuned Self-BLEU: 0.6525451316420547


---

In [83]:
collection_base = []
for i in range(30):
    baseline = pipe_base(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.9,
        top_p=0.95
    )
    collection_base.append(baseline[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

In [84]:
collection_base

['Once upon a time there was a fairy who was going to be there for you - one that seemed to be very nice, a fairy who told you about her magic.\n\n"You\'ll be looking after me until the end of my life."\n\nThe fairy was a man named Harry who was a friend of Hermione, and he was also one of the people who taught him the most wonderful magic.\n\n"Yes, I will," he said.\n\nHermione nodded, and the two of them went off to the Fairy Shop, and with a groan, she stepped into the carriage and started walking out.\n\nThe carriage stopped in front of it, and Hermione looked at it with confusion. It didn\'t matter how many times she tried to walk back, it would never come back. So she walked out of the carriage and looked around, looking for the carriage to stop.\n\nIt wasn\'t there.\n\nJust now, at least the fairy was smiling.\n\nHer heart was filled with',
 'Once upon a time there was a fairy of the woods, and then the fire went out and the fairy was killed." But the fire is not, as the writer 

In [86]:
collection_16655 = []
for i in range(30):
    ft_output_16655 = pipe_ckpt(
    prompt,
    max_new_tokens=200,
    min_new_tokens=100,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    no_repeat_ngram_size=3
    )
    collection_16655.append(ft_output_16655[0]["generated_text"])

In [87]:
collection_16655

['Once upon a time there was a fairy king who had two sons; but for the old woman, who knew her mother, she could not go to church, and for fear he had been in such a hurry to look for it all she knew the King was dead. The youth had been as hard at work, and that she had no heart, and there was no help for him at all; so when he got home, they went to the door, and she called out that he must go, and, after the Queen\'s death, so he said, "If I only had one son you must go first, and I will tell you how to save your mother\'s life. " So the wife opened the door of the church, which was so big that the boy could never get on, and when the old man came in, and the old father was at home, her life went on so well that he was so hungry that he couldn’t keep up. And when he had got home again the father was so',
 'Once upon a time there was a fairy king. He lived with his mother, who had two sons, and the youngest brother was called the King of the Three Princes. They were both very poor a

In [88]:
def distinct_n(text, n=2):
    tokens = text.lower().split()
    ngrams = list(zip(*[tokens[i:] for i in range(n)]))
    return len(set(ngrams)) / max(1, len(ngrams))

In [89]:
print("Distinct-2 (base):", distinct_n(collection_base[0], 2))
print("Distinct-2 (ft):  ", distinct_n(collection_16655[0], 2))

Distinct-2 (base): 0.9141104294478528
Distinct-2 (ft):   0.9333333333333333


---

In [92]:
def surprisal(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        loss = model(**inputs, labels=inputs["input_ids"]).loss
    return loss.item()

base_surprisal = surprisal(pipe_base.model, pipe_base.tokenizer, collection_base[0])
ft_surprisal_16655   = surprisal(pipe_base.model, pipe_base.tokenizer, collection_16655[0])

print("Surprisal (base):", base_surprisal)
print("Surprisal (ft):  ", ft_surprisal_16655)

Surprisal (base): 2.192286491394043
Surprisal (ft):   3.016834259033203


In [95]:
fairy_words = {"fairy", "king", "queen", "magic", "forest", "spell", "castle", "witch", "dragon", "prince", "princess"}

def fairy_score(text):
    tokens = set(text.lower().split())
    return len(tokens & fairy_words)

print("Fairy score (base):", fairy_score(collection_base[0]))
print("Fairy score (ft):  ", fairy_score(collection_16655[0]))

Fairy score (base): 1
Fairy score (ft):   2
